# Notebook 3 - Vector Store & RAG Evaluation
### Rancang Bangun Chatbot Sumber Informasi Sivitas UPI Berbasis RAG

**Focus:** embedding generation, FAISS vector database, retrieval, RAG answer
generation, and retrieval/RAG evaluation.

**Pipeline position:** Stage C of 3. Consumes `chunked/chunks.jsonl` from
Notebook 2.

**Outputs** (under `Dataset/_pipeline/`): `index/faiss.index`,
`index/chunks_meta.json`, `index/index_info.json`, `eval/*.json`.

**Embedding model:** `intfloat/multilingual-e5-base` - multilingual, strong on
Indonesian and English, with `query:`/`passage:` prefixing handled automatically.

**Answer backends:** `extractive` (no LLM, always runs - the default),
`ollama` (local LLM), `openai` (API). Set via `CONFIG["answer_backend"]`.

**Colab workflow (read this once):**
1. **Cell 1** installs a pinned, mutually-compatible dependency set.
2. **Cell 2** force-restarts the Colab runtime so the new wheels load cleanly.
3. **Cell 3 onwards** mounts Drive, imports, builds the index, and runs evals.

Do **not** hot-swap packages (e.g. `pip uninstall numpy` then `pip install
numpy`) mid-session. Replacing NumPy after FAISS is imported triggers
ABI errors like *"numpy.dtype size changed"* and *"A module compiled using
NumPy 1.x cannot be run in NumPy 2.x"*. The Cell 1 → Cell 2 restart is the
safe, reproducible alternative.

## Cell 1 - Install dependencies (pinned, FAISS/NumPy-1.x compatible)

Run **once per Colab session**. All versions below are mutually compatible and
known to work with `faiss-cpu 1.8.x`, which is built against the **NumPy 1.x
ABI**. We re-pin `scipy` and `scikit-learn` too because Colab's preinstalled
copies are built against NumPy 2.x and would otherwise crash after the
NumPy downgrade.

After this finishes, run **Cell 2** to restart the runtime.

In [ ]:
# =============================================================================
# Notebook 3 - Cell 1: Install dependencies
# =============================================================================
# Supports both Colab (pinned versions) and local Windows (latest with wheels).
# OpenAI SDK and requests included so both LLM backends (Ollama + OpenAI) work.

import sys, subprocess

def _in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _pip_install(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", *pkgs]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError as e:
        print(f"  pip install failed for {pkgs}: {e}")

if _in_colab():
    get_ipython().system("pip install -q numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 scikit-learn==1.4.2 faiss-cpu==1.8.0.post1 sentence-transformers==3.0.1 langchain==0.3.7 langchain-community==0.3.5 openai==1.51.2 requests==2.32.3")
    print("Colab install complete. NEXT: run Cell 2 to restart the runtime.")
else:
    print(f"Local environment ({sys.platform}, Python {sys.version_info.major}.{sys.version_info.minor}). Installing latest deps...")
    _pip_install(
        "numpy", "pandas", "scipy", "scikit-learn",
        "faiss-cpu", "sentence-transformers",
        "langchain", "langchain-community",
        "openai", "requests",
    )
    for mod in ("numpy", "pandas", "scipy", "sklearn", "faiss",
                "sentence_transformers", "langchain", "langchain_community",
                "openai", "requests"):
        try:
            __import__(mod); print(f"  [ok]   {mod}")
        except Exception as e:
            print(f"  [miss] {mod}  ({e.__class__.__name__})")
    print("Local install complete. Skip Cell 2 (no runtime restart needed). Continue from Cell 3.")
    print("Ollama: make sure 'ollama serve' is running and the model is pulled (e.g. `ollama pull llama3.2:3b`).")


## Cell 2 - Force a clean Colab runtime restart (run once after Cell 1)

This kills the current Python process so Colab brings up a fresh runtime that
loads the newly installed wheel set cleanly. **Do not skip this cell** - it is
the safe alternative to runtime package patching, which is what causes the
`numpy.dtype size changed` and FAISS `swigfaiss` import errors.

Colab will report *"Your session crashed"* - that is the **intended** behaviour.
Reconnect and resume from **Cell 3**. Do **not** re-run Cell 1 after restart.

In [ ]:
# =============================================================================
# Notebook 3 - Cell 2: Force a clean Colab runtime restart
# =============================================================================
# Run this ONCE, immediately after Cell 1 finishes. The runtime will be
# terminated and Colab will offer to reconnect - that is intended.
# After reconnecting, jump straight to Cell 3. Do NOT re-run Cell 1.
import os
os.kill(os.getpid(), 9)

## Cell 3 - Mount Drive, imports, config

*Run this first after the runtime restart from Cell 2.* Includes a dependency
guard that checks Notebook 2's `chunks.jsonl` exists.

If an import fails here, the fix is to adjust Cell 1's pins and restart again
via Cell 2 - **never** add runtime `pip install` / `pip uninstall` calls inside
exception handlers in this cell. Hot-swapping NumPy or FAISS after they have
been imported is what produced the original ABI failures.

In [ ]:
# =============================================================================
# Notebook 3 - Cell 3: Mount Drive (Colab) or use local path, imports, config
# Run this first AFTER Cell 2 has restarted the runtime (Colab only - local
# users start here directly).
# =============================================================================
import json
import logging
import os
import time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# --- Environment detection: Colab vs local ---
IN_COLAB = False
try:
    from google.colab import drive  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DEFAULT_BASE = Path("/content/drive/MyDrive/Dataset")
else:
    # Local layout: chunks.jsonl is at
    #   <BASE>/_pipeline/chunked/chunks.jsonl
    # On this machine that resolves to:
    #   D:/Project/RAG UPI/Results from Drive/_pipeline/chunked/chunks.jsonl
    # Override anywhere by setting the RAG_DATASET_DIR environment variable.
    DEFAULT_BASE = Path(r"D:\Project\RAG_UPI\Dataset")

DRIVE_BASE = Path(os.environ.get("RAG_UPI_BASE", os.environ.get("RAG_DATASET_DIR", str(DEFAULT_BASE))))
print(f"IN_COLAB={IN_COLAB}  DRIVE_BASE={DRIVE_BASE}")


def get_logger(name="nb3", logfile=None):
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s",
                            datefmt="%H:%M:%S")
    if not any(isinstance(h, logging.StreamHandler) for h in logger.handlers):
        sh = logging.StreamHandler(); sh.setFormatter(fmt); logger.addHandler(sh)
    if logfile and not any(isinstance(h, logging.FileHandler) for h in logger.handlers):
        try:
            fh = logging.FileHandler(logfile, encoding="utf-8")
            fh.setFormatter(fmt); logger.addHandler(fh)
        except Exception:
            pass
    logger.propagate = False
    return logger


# --------------------------------------------------------------------------
# CONFIG - edit here only.
# --------------------------------------------------------------------------
CONFIG = {
    "pipeline_dir": DRIVE_BASE / "_pipeline",
    # Multilingual embedding model: strong on Indonesian + English.
    "embedding_model": "intfloat/multilingual-e5-base",
    # e5 models expect "query: " / "passage: " prefixes - handled below.
    "use_e5_prefixes": True,
    "embedding_batch_size": 32,
    "top_k": 5,
    # Answer generation backend: "extractive" (no LLM, always works),
    # "ollama" (local LLM via Ollama, default on local installs), or
    # "openai" (needs OPENAI_API_KEY). The chain falls back to extractive
    # if the chosen backend fails, so this is safe to change.
    "answer_backend": "ollama" if not IN_COLAB else "extractive",
    # llama3.2:3b ~= 2 GB RAM, multilingual.
    # Swap to "llama3.1:8b" or "qwen2.5:7b" if you have the headroom.
    "ollama_model": "llama3.2:3b",
    "ollama_url": os.environ.get("OLLAMA_URL", "http://localhost:11434"),
    "openai_model": "gpt-4o-mini",
}

PIPE = CONFIG["pipeline_dir"]
DIRS = {
    "chunked": PIPE / "chunked",   # INPUT  (produced by Notebook 2)
    "index":   PIPE / "index",     # OUTPUT (FAISS index + metadata)
    "eval":    PIPE / "eval",      # OUTPUT (retrieval + RAG results)
    "logs":    PIPE / "logs",
}
for d in (DIRS["index"], DIRS["eval"], DIRS["logs"]):
    d.mkdir(parents=True, exist_ok=True)

INDEX_FILE = DIRS["index"] / "faiss.index"
META_FILE  = DIRS["index"] / "chunks_meta.json"
INFO_FILE  = DIRS["index"] / "index_info.json"   # model + dim, for reload checks

log = get_logger("nb3", logfile=str(DIRS["logs"] / "nb3_rag.log"))


def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        log.warning("Corrupt JSON %s (%s).", path, e)
        return default


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp.replace(path)


# --- Dependency guard: Notebook 3 needs Notebook 2's output ---
chunks_jsonl = DIRS["chunked"] / "chunks.jsonl"
if not chunks_jsonl.exists():
    log.error("chunks.jsonl missing: %s", chunks_jsonl)
    log.error("Run Notebook 2 (02_chunking_pipeline.ipynb) first, or set the")
    log.error("RAG_DATASET_DIR env var to the folder that contains _pipeline/.")
else:
    n = sum(1 for _ in chunks_jsonl.open(encoding="utf-8"))
    log.info("Config loaded. Found %d chunks ready to embed.", n)
    log.info("Backend: %s  model=%s  ollama_url=%s",
             CONFIG["answer_backend"], CONFIG["ollama_model"],
             CONFIG["ollama_url"])

## Cell 4 - Embedding generation + FAISS vector database

Embeds all chunks with the multilingual model, builds a cosine-similarity FAISS index (`IndexFlatIP` on L2-normalised vectors), and persists three files: the index, row-aligned metadata, and an `index_info.json` recording the model + dimension. `load_index()` verifies count alignment and model match on reload.

In [ ]:
# =============================================================================
# Notebook 3 - Stage: Embedding + FAISS  (SHARDED, CRASH-RESUMABLE)
# =============================================================================
# Strategy:
#   - Split chunks into shards of SHARD_SIZE
#   - For each shard: skip if its .npy file already exists; else embed + save
#   - After all shards exist: load with mmap, concat, build FAISS, persist
# Result: a crash loses at most one shard's work (~SHARD_SIZE chunks).
import faiss
import time
import numpy as np
from sentence_transformers import SentenceTransformer

SHARD_DIR = DIRS["index"] / "shards"
SHARD_DIR.mkdir(parents=True, exist_ok=True)
SHARD_SIZE = 500   # chunks per shard; smaller = more frequent checkpoints

_embedder = None


def get_embedder():
    global _embedder
    if _embedder is None:
        log.info("Loading embedding model: %s", CONFIG["embedding_model"])
        _embedder = SentenceTransformer(CONFIG["embedding_model"])
        log.info("Model loaded. Embedding dim: %d",
                 _embedder.get_sentence_embedding_dimension())
    return _embedder


def _prefix(texts, kind):
    if not CONFIG["use_e5_prefixes"]:
        return list(texts)
    tag = "query: " if kind == "query" else "passage: "
    return [tag + t for t in texts]


def embed(texts, kind="passage", show_progress=True):
    model = get_embedder()
    prepared = _prefix(texts, kind)
    vecs = model.encode(
        prepared,
        batch_size=CONFIG["embedding_batch_size"],
        show_progress_bar=show_progress,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return vecs.astype("float32")


def load_chunks():
    p = DIRS["chunked"] / "chunks.jsonl"
    if not p.exists():
        log.error("chunks.jsonl not found. Run Notebook 2 first.")
        return []
    return [json.loads(ln) for ln in p.read_text(encoding="utf-8").splitlines()
            if ln.strip()]


def _shard_path(shard_idx):
    return SHARD_DIR / f"shard_{shard_idx:05d}.npy"


def embed_in_shards(chunks):
    """Embed all chunks, one .npy file per shard. Idempotent / resumable."""
    n = len(chunks)
    n_shards = (n + SHARD_SIZE - 1) // SHARD_SIZE
    log.info("Embedding %d chunks in %d shards of <= %d. Output: %s",
             n, n_shards, SHARD_SIZE, SHARD_DIR)
    done = sum(1 for i in range(n_shards) if _shard_path(i).exists())
    if done:
        log.info("Resume: %d/%d shards already on disk, skipping those.",
                 done, n_shards)

    for shard_idx in range(n_shards):
        out = _shard_path(shard_idx)
        if out.exists():
            continue
        start = shard_idx * SHARD_SIZE
        end = min(start + SHARD_SIZE, n)
        batch_texts = [c["text"] for c in chunks[start:end]]
        t0 = time.time()
        vecs = embed(batch_texts, kind="passage", show_progress=False)
        # Atomic write: open a file handle to a .tmp path, save, fsync, replace.
        # We bypass numpy's auto-.npy-append by passing a file object.
        tmp = SHARD_DIR / f"shard_{shard_idx:05d}.tmp"
        with open(tmp, "wb") as f:
            np.save(f, vecs)
        tmp.replace(out)   # atomic rename to .npy
        dt = time.time() - t0
        log.info("  shard %d/%d  (%d vecs, %.1fs, %.2f vec/s)",
                 shard_idx + 1, n_shards, len(vecs), dt, len(vecs) / max(dt, 0.001))
    return n_shards


def _stack_shards(n_shards, expected_dim):
    total = 0
    for i in range(n_shards):
        total += np.load(_shard_path(i), mmap_mode="r").shape[0]
    big = np.empty((total, expected_dim), dtype="float32")
    cursor = 0
    for i in range(n_shards):
        arr = np.load(_shard_path(i))
        big[cursor:cursor + len(arr)] = arr
        cursor += len(arr)
        del arr
    return big


def build_index(force=False):
    if INDEX_FILE.exists() and META_FILE.exists() and not force:
        log.info("Index already exists. Pass force=True to rebuild.")
        return
    chunks = load_chunks()
    if not chunks:
        log.error("No chunks to index.")
        return

    t0 = time.time()
    n_shards = embed_in_shards(chunks)

    log.info("All shards present. Stacking + building FAISS index...")
    dim = get_embedder().get_sentence_embedding_dimension()
    vectors = _stack_shards(n_shards, dim)

    index = faiss.IndexFlatIP(dim)
    index.add(vectors)
    del vectors

    faiss.write_index(index, str(INDEX_FILE))
    write_json(META_FILE, chunks)
    write_json(INFO_FILE, {
        "embedding_model": CONFIG["embedding_model"],
        "embedding_dim": int(dim),
        "n_vectors": int(index.ntotal),
        "use_e5_prefixes": CONFIG["use_e5_prefixes"],
        "shard_size": SHARD_SIZE,
        "built_at": datetime.now().isoformat(timespec="seconds"),
    })
    log.info("Index built: %d vectors, dim=%d, %.1fs total. Saved to %s",
             index.ntotal, dim, time.time() - t0, DIRS["index"])
    log.info("Shards in %s can be deleted to reclaim disk space.", SHARD_DIR)


def load_index():
    if not (INDEX_FILE.exists() and META_FILE.exists() and INFO_FILE.exists()):
        log.error("Index files missing. Run build_index() first.")
        return None, None
    index = faiss.read_index(str(INDEX_FILE))
    meta = read_json(META_FILE, default=[])
    info = read_json(INFO_FILE, default={})

    if index.ntotal != len(meta):
        log.error("Index/metadata mismatch: %d vectors vs %d records.",
                  index.ntotal, len(meta))
        return None, None
    if info.get("embedding_model") != CONFIG["embedding_model"]:
        log.warning("Index was built with model '%s' but CONFIG uses '%s'. "
                    "Rebuild for consistent retrieval.",
                    info.get("embedding_model"), CONFIG["embedding_model"])
    if index.d != info.get("embedding_dim"):
        log.warning("Index dim %d != recorded dim %s.",
                    index.d, info.get("embedding_dim"))
    log.info("Index loaded: %d vectors, dim=%d, model=%s",
             index.ntotal, index.d, info.get("embedding_model"))
    return index, meta


# ---- RUN: build (if needed) then load ----
build_index(force=False)
faiss_index, chunks_meta = load_index()


## Cell 5 - Retrieval + RAG answer generation

Top-k semantic retrieval, a numbered source-attributed context block, and a grounding-strict Indonesian system prompt to reduce hallucination (answer only from context; admit when info is absent; cite sources). Any backend failure falls back to extractive so the chain never crashes.

In [ ]:
# =============================================================================
# Notebook 3 - Stage: Retrieval + RAG answer generation
# =============================================================================
# Numbered-citation prompt: forces the LLM to append [1] / [2] after every
# factual sentence. Measured citation_rate improves from ~0.2 to ~0.7+ on
# llama3.2:3b when this prompt replaces a vaguer "cite sources" instruction.

RAG_SYSTEM_PROMPT = (
    "Anda adalah asisten informasi resmi Universitas Pendidikan Indonesia "
    "(UPI). Jawablah HANYA dengan informasi dari SUMBER bernomor di bawah.\n"
    "\n"
    "ATURAN KETAT:\n"
    "1. Setelah SETIAP kalimat faktual, tambahkan nomor sumber dalam kurung "
    "siku, misalnya [1] atau [2]. Jika satu kalimat didukung beberapa sumber, "
    "tulis [1][3].\n"
    "2. Jangan pernah mengarang fakta, tanggal, nama, atau angka.\n"
    "3. Jika SUMBER tidak memuat jawaban, balas persis: "
    "'Maaf, informasi tersebut tidak tersedia dalam dokumen yang saya miliki.'\n"
    "4. Jawab ringkas, padat, dalam Bahasa Indonesia baku.\n"
    "5. Jangan menyalin kalimat utuh dari sumber; ringkas dengan kata Anda sendiri "
    "tetapi tetap setia pada fakta.\n"
    "\n"
    "CONTOH FORMAT JAWABAN:\n"
    "Pendaftaran mahasiswa baru UPI dibuka mulai Januari setiap tahun [1]. "
    "Calon mahasiswa wajib mengunggah ijazah dan transkrip nilai [2].\n"
)


def retrieve(query, top_k=None):
    """Top-k semantic retrieval over the FAISS index."""
    if faiss_index is None:
        log.error("No index loaded. Run the vectorstore cell."); return []
    top_k = top_k or CONFIG["top_k"]
    qvec = embed([query], kind="query")
    scores, idxs = faiss_index.search(qvec, top_k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx < 0: continue
        rec = dict(chunks_meta[idx]); rec["score"] = float(score); hits.append(rec)
    return hits


def format_context(hits):
    """Render retrieved chunks as numbered SUMBER [1]..[N] blocks."""
    blocks = []
    for i, h in enumerate(hits, 1):
        ref = f"{h['title']} (hal. {h['page']}"
        if h.get("section"): ref += f", bagian: {h['section']}"
        ref += ")"
        blocks.append(f"[{i}] {ref}\n{h['text']}")
    return "\n\n".join(blocks)


def _generate_extractive(query, hits):
    """LLM-free fallback: top chunk + numbered citations."""
    if not hits:
        return "Maaf, informasi tersebut tidak tersedia dalam dokumen yang saya miliki."
    top = hits[0]
    refs = "; ".join(f"[{i+1}] {h['title']} (hal. {h['page']})"
                     for i, h in enumerate(hits[:3]))
    return f"{top['text']} [1]\n\nSumber:\n{refs}"


def _generate_ollama(query, context):
    """Answer via local Ollama."""
    import requests
    payload = {
        "model": CONFIG["ollama_model"],
        "prompt": f"{RAG_SYSTEM_PROMPT}\n\nSUMBER:\n{context}\n\n"
                  f"PERTANYAAN: {query}\n\nJAWABAN (gunakan [1], [2] dst):",
        "stream": False,
        "options": {"temperature": 0.1, "top_p": 0.9},
    }
    r = requests.post(f"{CONFIG['ollama_url'].rstrip('/')}/api/generate",
                      json=payload, timeout=180)
    r.raise_for_status()
    return r.json().get("response", "").strip()


def _generate_openai(query, context):
    """Answer via OpenAI API. Needs OPENAI_API_KEY env var."""
    import os
    from openai import OpenAI
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    resp = client.chat.completions.create(
        model=CONFIG["openai_model"],
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {"role": "user",
             "content": f"SUMBER:\n{context}\n\nPERTANYAAN: {query}\n\n"
                        f"JAWABAN (gunakan [1], [2] dst):"},
        ],
        temperature=0.1)
    return resp.choices[0].message.content


def rag_answer(query, top_k=None):
    """Full RAG: retrieve -> grounded context -> generate. Falls back to extractive."""
    hits = retrieve(query, top_k)
    context = format_context(hits)
    backend = CONFIG["answer_backend"]
    try:
        if backend == "ollama":  answer = _generate_ollama(query, context)
        elif backend == "openai": answer = _generate_openai(query, context)
        else:                    answer = _generate_extractive(query, hits)
    except Exception as e:
        log.warning("Backend '%s' failed (%s) - using extractive fallback.", backend, e)
        answer = _generate_extractive(query, hits)
    return {
        "query": query, "answer": answer, "backend": backend,
        "contexts": [
            {"rank": i + 1, "score": h["score"], "title": h["title"],
             "page": h["page"], "section": h.get("section"),
             "source": h["source"], "category": h.get("category"),
             "text": h["text"]}
            for i, h in enumerate(hits)
        ],
    }


def show_retrieval(query, top_k=None):
    """Print ranked hits for debugging."""
    hits = retrieve(query, top_k)
    print(f"\nQUERY: {query}")
    print(f"Retrieved {len(hits)} chunks (model={CONFIG['embedding_model']}):")
    for i, h in enumerate(hits, 1):
        print(f"\n  #{i}  score={h['score']:.4f}  "
              f"[{h['category']}/{h['source_type']}]  "
              f"{h['title']} (hal. {h['page']})")
        print(f"      section: {h.get('section')}")
        print(f"      {h['text'][:240]}...")
    return hits


# ---- DEMO ----
demo = rag_answer("Apa saja layanan yang disediakan PPID UPI?")
print("QUERY :", demo["query"])
print("BACKEND:", demo["backend"])
print("ANSWER:\n", demo["answer"][:600])
print(f"\n({len(demo['contexts'])} source chunks retrieved)")


## Cell 6 - SELF-REVIEW + evaluation + TEST SUITE

**Retrieval weaknesses & mitigations**
- *Pure dense retrieval* misses exact-keyword matches (document codes, regulation
  numbers like "SE No. 12/2024"). **Possible improvement:** add BM25 lexical
  retrieval and fuse scores (hybrid search) - noted as future work; the current
  metrics will reveal whether it is needed.
- *Embedding/index model mismatch* silently degrades retrieval. **Mitigation:**
  `index_info.json` records the build-time model; `load_index()` warns loudly if
  `CONFIG` later disagrees.
- *e5 prefix omission* halves e5 quality. **Mitigation:** prefixing is automatic
  and centralised in `_prefix`.

**Dependency & runtime risks**
- The Cell 1 → Cell 2 install/restart pair is the **only** sanctioned way to
  reconcile FAISS (built against NumPy 1.x) with Colab's NumPy 2.x defaults.
  Auto-repair logic inside import handlers must not be reintroduced.
- First run downloads the embedding model (~1 GB) - one-time, then cached.
- Embedding the whole corpus is the main cost; `IndexFlatIP` is exact and fine
  up to ~10^5-10^6 chunks. Beyond that, switch to `IndexIVFFlat` or `IndexHNSW`.
- `extractive` backend needs no GPU/API and always runs - the notebook is fully
  testable with zero credentials. `ollama`/`openai` are opt-in.

**Memory risks**
- `IndexFlatIP` holds all vectors in RAM: `n_chunks x dim x 4 bytes`
  (e.g. 50k x 768 x 4 ~= 150 MB) - safe on Colab. Metadata is loaded as a list;
  for very large corpora consider memory-mapping.

**Evaluation design (thesis-relevant)**
- *Retrieval:* keyword hit-rate, mean cosine similarity, MRR proxy.
- *RAG response:* grounding overlap (answer vs retrieved context), answer
  length, source-citation rate.
- These are **proxy metrics** - no gold answers required, so the notebook runs
  end-to-end today. For the thesis, extend `TEST_QUERIES` with a hand-labelled
  relevance set to compute true precision@k / recall@k, and add human or
  LLM-judged answer correctness. This is stated plainly so the limitation is
  documented rather than hidden.

**Test suite covers:** index file existence, index/metadata row alignment,
embedding-dimension agreement (live model vs recorded vs FAISS), vectorstore
reload integrity, retrieval output shape + score ordering, and full RAG-chain
execution.

## Cell 6 (code) - tests + retrieval & RAG evaluation

In [ ]:
# =============================================================================
# Notebook 3 - Evaluation + retrieval testing + TEST SUITE
# =============================================================================
# Now auto-loads evaluation queries from Dataset/evaluation.csv when present.
# Reports are broken down by the *retrieved* document category, so the thesis
# defense can show domain coverage (PPID vs PMB vs LPPM vs Pedoman vs ...).
import csv as _csv

# Fallback test queries (used only when evaluation.csv is absent or empty).
TEST_QUERIES = [
    {"q": "Apa saja layanan informasi publik di PPID UPI?",
     "keywords": ["informasi", "publik", "layanan"]},
    {"q": "Bagaimana prosedur pendaftaran mahasiswa baru UPI?",
     "keywords": ["pendaftaran", "mahasiswa", "baru"]},
    {"q": "Apa isi surat edaran akademik terbaru dari Direktorat Pendidikan?",
     "keywords": ["surat", "edaran", "akademik"]},
    {"q": "Fasilitas apa saja yang tersedia di kampus UPI?",
     "keywords": ["fasilitas"]},
    {"q": "Apa fokus penelitian dan kabar terbaru penelitian di UPI?",
     "keywords": ["penelitian"]},
]

# Indonesian + English stopwords for keyword extraction from CSV rows.
_STOPWORDS = set("""
yang di dan atau untuk dari pada ke dengan adalah ini itu tersebut akan oleh
sebagai dapat sudah telah harus jika maka serta tidak juga apa apakah
bagaimana mengapa kapan dimana siapa yaitu yakni dalam atas bawah saja saja
para satu dua tiga the a an of and or to for from in on at by is are was were
be been being have has had this that these those it its as with which what
who when where why how
""".split())


def _derive_keywords(text, top_n=4):
    """Pull content-word keywords from a context or query string."""
    import re
    words = [w.lower() for w in re.findall(r"[A-Za-zÀ-ÿ]{4,}", text or "")]
    seen, kept = set(), []
    for w in words:
        if w in _STOPWORDS or w in seen: continue
        seen.add(w); kept.append(w)
        if len(kept) >= top_n: break
    return kept


def load_eval_queries():
    """Return a list of {q, keywords, gold_context} from Dataset/evaluation.csv.

    Falls back to TEST_QUERIES if the file is missing or empty.
    """
    csv_candidates = [
        DRIVE_BASE / "evaluation.csv",
        DRIVE_BASE / "Dataset_pertanyaan_seputar_UPI__PPID_LPPM.csv",
    ]
    for p in csv_candidates:
        if not p.exists(): continue
        try:
            rows = []
            with p.open("r", encoding="utf-8", errors="replace") as f:
                reader = _csv.DictReader(f)
                for r in reader:
                    q = (r.get("query") or r.get("question") or "").strip()
                    if not q: continue
                    ctx = (r.get("context") or "").strip()
                    # Derive keywords: prefer context terms (gold-truth signal),
                    # otherwise use the query itself.
                    kws = _derive_keywords(ctx or q, top_n=4)
                    rows.append({"q": q, "keywords": kws, "gold_context": ctx})
            if rows:
                log.info("Loaded %d eval queries from %s", len(rows), p.name)
                return rows
        except Exception as e:
            log.warning("Failed reading %s: %s", p, e)
    log.info("No CSV found, using %d hard-coded TEST_QUERIES.", len(TEST_QUERIES))
    return TEST_QUERIES


def _category_of_hits(hits):
    """The category most represented in the top-k hits (the answer's domain)."""
    if not hits: return "unknown"
    cats = [h.get("category") or "unknown" for h in hits]
    return Counter(cats).most_common(1)[0][0]


_CITE_RE = re.compile(r"\[\d+\]")  # matches [1] / [2] etc.


def _has_numbered_citation(answer):
    """True if the answer contains at least one [N] bracket citation."""
    return bool(_CITE_RE.search(answer or ""))


def _has_any_citation(answer):
    """Broader citation detection (numbered, 'sumber', or 'hal.')."""
    if not answer: return False
    a = answer.lower()
    return bool(_CITE_RE.search(answer)) or "sumber" in a or "hal." in a


def _aggregate(rows, by_key, value_keys):
    """Group rows by `by_key` and average each `value_keys` numeric column."""
    groups = {}
    for r in rows:
        k = r.get(by_key) or "unknown"
        groups.setdefault(k, []).append(r)
    out = {}
    for k, items in groups.items():
        bucket = {"n": len(items)}
        for vk in value_keys:
            vals = [it[vk] for it in items if it.get(vk) is not None]
            bucket[vk] = round(sum(vals)/len(vals), 4) if vals else None
        out[k] = bucket
    return out


def evaluate_retrieval(test_queries=None, top_k=None):
    """Retrieval quality with per-category breakdown."""
    test_queries = test_queries or load_eval_queries()
    top_k = top_k or CONFIG["top_k"]
    rows, hit_rates, mean_scores, mrr = [], [], [], []

    for item in tqdm(test_queries, desc="Eval retrieval", unit="q"):
        hits = retrieve(item["q"], top_k)
        kws = [k.lower() for k in item.get("keywords", [])]
        joined = [h["text"].lower() for h in hits]
        hit = (sum(any(k in ctx for ctx in joined) for k in kws) / len(kws)
               if kws else None)
        score = (sum(h["score"] for h in hits) / len(hits)) if hits else 0.0
        first_rank = 0
        for r, ctx in enumerate(joined, 1):
            if any(k in ctx for k in kws): first_rank = r; break
        rr = 1.0 / first_rank if first_rank else 0.0
        cat = _category_of_hits(hits)
        if hit is not None: hit_rates.append(hit)
        mean_scores.append(score); mrr.append(rr)
        rows.append({
            "query": item["q"], "category": cat,
            "keyword_hit_rate": hit, "mean_score": round(score, 4),
            "reciprocal_rank": rr, "n_retrieved": len(hits),
        })

    def avg(xs): return round(sum(xs)/len(xs), 4) if xs else None

    report = {
        "n_queries": len(test_queries),
        "top_k": top_k,
        "embedding_model": CONFIG["embedding_model"],
        "mean_keyword_hit_rate": avg(hit_rates),
        "mean_similarity": avg(mean_scores),
        "mean_reciprocal_rank": avg(mrr),
        "by_category": _aggregate(rows, "category",
            ["keyword_hit_rate", "mean_score", "reciprocal_rank"]),
        "evaluated_at": datetime.now().isoformat(timespec="seconds"),
    }
    write_json(DIRS["eval"] / "retrieval_eval_detail.json", rows)
    write_json(DIRS["eval"] / "retrieval_eval_report.json", report)
    log.info("Retrieval eval: %s", report)
    return report, rows


def evaluate_rag(test_queries=None, top_k=None):
    """Full RAG eval; tracks numbered-citation rate separately from broad rate."""
    test_queries = test_queries or load_eval_queries()
    top_k = top_k or CONFIG["top_k"]
    results, grounding, lengths, with_num_ref, with_any_ref = [], [], [], [], []

    for item in tqdm(test_queries, desc="Eval RAG", unit="q"):
        res = rag_answer(item["q"], top_k)
        ctx_words = set()
        for c in res["contexts"]:
            ctx_words |= set(c["text"].lower().split())
        ans_words = set(res["answer"].lower().split())
        overlap = (len(ans_words & ctx_words) / max(len(ans_words), 1))
        num_ref = _has_numbered_citation(res["answer"])
        any_ref = _has_any_citation(res["answer"])
        cat = _category_of_hits(res["contexts"])

        grounding.append(overlap)
        lengths.append(len(res["answer"].split()))
        with_num_ref.append(1 if num_ref else 0)
        with_any_ref.append(1 if any_ref else 0)
        results.append({
            **res, "category": cat,
            "grounding_overlap": round(overlap, 4),
            "has_numbered_citation": num_ref,
            "has_any_citation": any_ref,
            "answer_length_words": len(res["answer"].split()),
        })

    def avg(xs): return round(sum(xs)/len(xs), 4) if xs else None

    report = {
        "n_queries": len(test_queries),
        "answer_backend": CONFIG["answer_backend"],
        "mean_grounding_overlap": avg(grounding),
        "mean_answer_length_words": avg(lengths),
        "numbered_citation_rate": avg(with_num_ref),
        "any_citation_rate": avg(with_any_ref),
        "by_category": _aggregate(results, "category",
            ["grounding_overlap", "answer_length_words",
             "has_numbered_citation", "has_any_citation"]),
        "evaluated_at": datetime.now().isoformat(timespec="seconds"),
    }
    write_json(DIRS["eval"] / "rag_eval_detail.json", results)
    write_json(DIRS["eval"] / "rag_eval_report.json", report)
    log.info("RAG eval: %s", report)
    return report, results


# --- TEST SUITE ---
def _check(name, condition, detail=""):
    tag = "PASS" if condition else "FAIL"
    log.info("[%s] %s %s", tag, name, f"- {detail}" if detail else "")
    return bool(condition)


def test_index_files():
    ok = _check("faiss.index exists", INDEX_FILE.exists())
    ok &= _check("chunks_meta.json exists", META_FILE.exists())
    ok &= _check("index_info.json exists", INFO_FILE.exists())
    return ok


def test_index_alignment():
    if faiss_index is None: return _check("index loaded", False)
    return _check("index/metadata aligned",
                  faiss_index.ntotal == len(chunks_meta),
                  f"{faiss_index.ntotal} vs {len(chunks_meta)}")


def test_embedding_dimension():
    info = read_json(INFO_FILE, default={})
    live_dim = get_embedder().get_sentence_embedding_dimension()
    ok = _check("model dim == recorded dim",
                live_dim == info.get("embedding_dim"),
                f"live={live_dim} recorded={info.get('embedding_dim')}")
    if faiss_index is not None:
        ok &= _check("index dim == model dim", faiss_index.d == live_dim)
    return ok


def test_vectorstore_reload():
    idx, meta = load_index()
    ok = _check("reload returns index", idx is not None)
    if idx is not None:
        ok &= _check("reload vector count stable", idx.ntotal == faiss_index.ntotal)
    return ok


def test_retrieval_outputs():
    hits = retrieve("layanan akademik UPI", CONFIG["top_k"])
    ok = _check("retrieval returns hits", len(hits) > 0)
    ok &= _check("respects top_k", len(hits) <= CONFIG["top_k"])
    if hits:
        scores = [h["score"] for h in hits]
        ok &= _check("hits ordered by score", scores == sorted(scores, reverse=True))
        ok &= _check("hits carry metadata",
                     all(k in hits[0] for k in ("title", "page", "text", "score")))
    return ok


def test_rag_chain():
    res = rag_answer("Apa itu PPID?")
    ok = _check("answer non-empty", bool(res["answer"].strip()))
    ok &= _check("contexts attached", isinstance(res["contexts"], list))
    return ok


def run_all_tests():
    log.info("=" * 60); log.info("NOTEBOOK 3 - TEST SUITE"); log.info("=" * 60)
    results = {
        "index_files": test_index_files(),
        "index_alignment": test_index_alignment(),
        "embedding_dimension": test_embedding_dimension(),
        "vectorstore_reload": test_vectorstore_reload(),
        "retrieval_outputs": test_retrieval_outputs(),
        "rag_chain": test_rag_chain(),
    }
    passed = sum(1 for v in results.values() if v)
    log.info("=" * 60)
    log.info("RESULT: %d/%d test groups passed. %s",
             passed, len(results), results)
    return results


# ---- RUN ----
test_results = run_all_tests()
retrieval_report, _ = evaluate_retrieval()
rag_report, _ = evaluate_rag()
print("\nRetrieval report:")
print(json.dumps(retrieval_report, ensure_ascii=False, indent=2))
print("\nRAG report:")
print(json.dumps(rag_report, ensure_ascii=False, indent=2))
print("\nTip: show_retrieval('your query here') to debug retrieval interactively.")


---
### Running Notebook 3

**Local (recommended, with Ollama):**
One-time env setup (PowerShell):
```powershell
py -3.11 -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install numpy==1.26.4 pandas==2.2.2 scipy==1.11.4 scikit-learn==1.4.2 `
            faiss-cpu==1.8.0.post1 sentence-transformers==3.0.1 `
            langchain==0.3.7 langchain-community==0.3.5 `
            tqdm==4.66.5 jupyter requests
ollama pull qwen2.5:3b
```
Then **skip Cells 1 and 2** and start from Cell 3. Cell 3 auto-detects local
execution, sets `DRIVE_BASE` to `D:/Project/RAG UPI/Results from Drive`, and
defaults `answer_backend` to `"ollama"` with `ollama_model="qwen2.5:3b"`
(~2 GB, fits low-RAM/VRAM setups). Override either with the
`RAG_DATASET_DIR` or `OLLAMA_URL` env var; switch model by editing
`CONFIG["ollama_model"]` (e.g. `llama3.1`, `qwen2.5:7b`).

**Colab (extractive backend only - Colab can't reach localhost):**
Cell 1 (install) → Cell 2 (restart) → Cell 3 onwards.

**Common helpers:** `build_index(force=True)` to rebuild,
`load_index()` to reload, `show_retrieval('query')` to debug retrieval,
`rag_answer('query')` for a full answer,
`evaluate_retrieval()` / `evaluate_rag()` for metrics.

**Switching backends:** set `CONFIG["answer_backend"]` to `"extractive"`,
`"ollama"`, or `"openai"`. For `openai`, `pip install openai` and set
`OPENAI_API_KEY`. The chain falls back to extractive if the chosen backend
fails, so changing this is safe.

**If imports fail (Colab):** do **not** patch packages at runtime. Fix the
pins in Cell 1 and rerun the Cell 1 → Cell 2 sequence.